# Mid-Training, Annealing, and Continual Pretraining: Pretraining Is Not the End

> **Previous recap**: you learned how to train an LLM (Part 5), scaling laws (Part 9), data pipelines (Part 10), and LoRA fine-tuning (Part 11).
>
> Goal for this part: understand a key 2024 training mindset: **pretraining is a starting point, not the finish line.**

Core questions:
1. If you want to continue training halfway through, what do you do? -> **WSD schedule**
2. How do you "sprint" in the last stage to push loss down? -> **annealing**
3. How do you turn a general model into a domain expert? -> **continual pretraining (CPT)**
4. How do you learn new things without forgetting old things? -> **freeze + data mixing (LoRA helps)**


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

## 1. One question that motivates the whole part

Suppose you trained a 7B model with a cosine LR schedule for 1T tokens.
After finishing, you realize: if you add another 500B tokens, loss should still go down.

But then a problem appears:

```
Cosine schedule requires you to commit to total steps T up front.
You already reached T, so LR decayed close to 0.
Now you want to continue training, but LR is "out of fuel".
```

This is a weakness of classic cosine.
In 2024, the MiniCPM team proposed a practical fix: **WSD schedule**.


## 2. WSD schedule: make training "continuable"

### 2.1 Three phases: warmup -> stable -> decay

WSD splits training into three parts:

```
         Warmup    |     Stable (constant)   |   Decay (anneal)
  lr_max ----\      |                        |                  /   \     |                        |                  /     \    |                        |            0 ----/-------\---+------------------------+-----------\---
       first 5%      middle 80-85%                 last 10-15%
```

| Phase | LR behavior | Portion | Purpose |
|:---|:---|:---|:---|
| Warmup | 0 -> max | first ~5% | avoid unstable large steps at the beginning |
| Stable | **constant** | ~80-85% | train at full speed; can stop anytime |
| Decay | max -> 0 | last ~10-15% | anneal and converge |

Key innovation is the **Stable** phase: LR stays constant.
So at any checkpoint in Stable you still have full LR, and you can:
- decide later when to start decay
- keep training longer (overtraining) without pre-committing to a horizon


In [ ]:
# === Cosine vs WSD manualCompare ===
print("=== Cosine vs WSD：Key LR Compare ===")
print()

T = 1000
warmup = 50
stable_end = 850
eta_max = 0.01

def cosine_lr(t):
    """Cosine: warmup """
    if t < warmup:
        return eta_max * t / warmup
    progress = (t - warmup) / (T - warmup)
    return eta_max * 0.5 * (1 + np.cos(np.pi * progress))

def wsd_lr(t):
    """WSD: warmup -> stable(stable) -> decay"""
    if t < warmup:
        return eta_max * t / warmup
    elif t < stable_end:
        return eta_max  # stable！
    else:
        progress = (t - stable_end) / (T - stable_end)
        return eta_max * (0.001 ** progress)

print(f"{' ':>6s}  {'phase':>8s}  {'Cosine LR':>12s}  {'WSD LR':>12s}  {' '}")
print("-" * 60)

for t in [0, 50, 200, 500, 700, 850, 900, 950, 999]:
    if t < warmup:
        phase = "Warmup"
    elif t < stable_end:
        phase = "Stable"
    else:
        phase = "Decay"
    
    cos = cosine_lr(t)
    wsd = wsd_lr(t)
    diff = f"WSD Cosine {wsd/cos:.1f}x" if cos > 0 else "—"
    print(f"{t:>6d}  {phase:>8s}  {cos:>12.6f}  {wsd:>12.6f}  {diff}")

print()
print("KeyObservation：")
print(" 500 ：Cosine 0.004,WSD 0.01")
print(" 700 ：Cosine 0.001,WSD ")
print(" -> Cosine training ,WSD ")
print()
print("Conclusion： resume training？WSD .Cosine？LR .")

In [ ]:
# === ：Cosine vs WSD ===
steps = np.arange(T)
cos_lrs = [cosine_lr(t) for t in steps]
wsd_lrs = [wsd_lr(t) for t in steps]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

#  ： training
ax = axes[0]
ax.plot(steps, cos_lrs, 'b-', lw=1.5, label='Cosine')
ax.plot(steps, wsd_lrs, 'r-', lw=1.5, label='WSD')
ax.axvspan(0, warmup, alpha=0.1, color='green', label='Warmup')
ax.axvspan(warmup, stable_end, alpha=0.05, color='blue', label='Stable')
ax.axvspan(stable_end, T, alpha=0.1, color='red', label='Decay')
ax.set_xlabel('Step')
ax.set_ylabel('Learning Rate')
ax.set_title('Cosine vs WSD')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

#  ：resume training 
ax = axes[1]
ext = np.arange(600, 1200)
cos_ext = [cosine_lr(min(t, T-1)) for t in ext]  # Cosine 0
wsd_ext = [wsd_lr(t) if t < T else eta_max * (0.001 ** ((t - stable_end) / 200)) for t in ext]
ax.plot(ext, cos_ext, 'b-', lw=1.5, label='Cosine (continue training)')
ax.plot(ext, wsd_ext, 'r-', lw=1.5, label='WSD (continue training)')
ax.axvline(x=600, color='green', lw=1.5, label='Continuation start')
ax.set_xlabel('Step')
ax.set_ylabel('Learning Rate')
ax.set_title('Continuation: cosine is near zero')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print(" ：WSD 85% ")
print(" ：resume training Cosine LR,WSD ")


### 2.2 Overtraining + WSD becomes a 2024 standard recipe

Recall from scaling laws: LLaMA 3 8B used a huge D/N ratio (1875x), far above Chinchilla's 20x.
This is "overtraining": small model, massive data, cheaper inference.

But cosine does not support indefinite overtraining: you must decide T ahead of time.

WSD fixes this:

```
Classic (cosine): pick data amount -> pick total steps T -> cosine decays -> done
Modern (WSD): train in stable forever -> stop anytime -> run decay at the end
```

So overtraining becomes operational.


---

## 3. Annealing: why the last 10% of training matters

WSD's Decay phase is also called annealing.
The term comes from metallurgy: heat a metal and cool slowly to reach a stronger, more ordered structure.

Training analogy: as LR decreases, parameters "settle" into a better region.

### 3.1 Shooting-range intuition

```
Stable (large LR):
  shooting from 100m. shots land around the bullseye but keep oscillating.

Decay (LR decreasing):
  you walk closer: 50m, 30m, 10m.
  each update is smaller but more precise.
  and stable phase already found the rough direction.
  -> accuracy improves fast.
```

Three takeaways:
1. direction is largely found in Stable
2. smaller steps in Decay are more precise
3. mixing higher-quality data during Decay can guide convergence


In [ ]:
# === annealingphase Loss ===
print("=== annealing：Loss finalphase ===")
print()

np.random.seed(42)
total_steps = 1000
stable_end = 850

steps = np.arange(total_steps)
loss = np.zeros(total_steps)

# Stable phase：loss 
for t in range(stable_end):
    progress = (t + 1) / stable_end
    loss[t] = 3.0 * (progress ** (-0.05)) + np.random.normal(0, 0.02)

# Decay phase：loss (annealing )
final_stable = loss[stable_end - 1]
for t in range(stable_end, total_steps):
    progress = (t - stable_end) / (total_steps - stable_end)
    drop = 0.15 * (1 - np.exp(-progress * 5))  #  
    loss[t] = final_stable - drop + np.random.normal(0, 0.01)

# manual 
stable_avg = np.mean(loss[800:850])
decay_avg = np.mean(loss[-50:])
improvement = stable_avg - decay_avg

stable_per_step = (4.0 - stable_avg) / stable_end
decay_per_step = improvement / (total_steps - stable_end)

print(f"Stable loss: {stable_avg:.4f}")
print(f"Decay loss: {decay_avg:.4f}")
print(f"annealing : {improvement:.4f} ({improvement/stable_avg*100:.1f}%)")
print()
print(f"Stable : {stable_per_step:.6f}")
print(f"Decay : {decay_per_step:.6f}")
print(f"annealing Stable {decay_per_step/stable_per_step:.1f}x")
print()
print("Key：annealing 15% , ！")

In [ ]:
# === annealing ===
fig, axes = plt.subplots(2, 1, figsize=(10, 7))

#  ： loss 
ax = axes[0]
ax.plot(steps, loss, 'b-', lw=0.8, alpha=0.7)
ax.axvline(x=stable_end, color='red', ls='--', lw=1.5, label=f'Decay starts (step {stable_end})')
ax.axvspan(0, 50, alpha=0.1, color='green', label='Warmup')
ax.axvspan(50, stable_end, alpha=0.05, color='blue', label='Stable')
ax.axvspan(stable_end, 1000, alpha=0.1, color='red', label='Decay annealing')
ax.set_ylabel('Loss')
ax.set_title('WSD training loss')
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

#  ： decay
ax = axes[1]
ax.plot(steps[750:], loss[750:], 'b-', lw=1.5)
ax.axvline(x=stable_end, color='red', ls='--', lw=2, label='Decay starts')
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.set_title('Zoom: loss drops faster during annealing')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
print("Note decay phase( ) —— annealing .")


---

## 4. Mid-training data strategy: what data should you feed during annealing?

In annealing, not only LR changes. **Data distribution can change too.**

Intuition: like exam sprint week: you stop grinding random practice and focus on high-quality notes.

MiniCPM ablations compared three strategies:

| Strategy | Stable data | Decay data | Result |
|:---|:---|:---|:---|
| A: no switch | CC+Code+Books | same | baseline |
| B: anneal then SFT | same | same, then do SFT after | better |
| C: **mix during anneal** | CC+Code+Books | add high-quality data during Decay | **best** |

Why C wins:
- early Decay LR still sizable -> can absorb new patterns
- late Decay LR small -> "weld" the patterns into weights
- LR decay and data shift work together: learn + consolidate


In [ ]:
# === Decay phasedata manual ===
print("=== annealingphasedata ===")
print()

#  ：2.4B model, 1.2T tokens
# Stable: 1T( data) Decay: 200B( + )
decay_total = 200  # B tokens

schemes = [
    (" ",        100,  0,  0,  0),
    (" ",       70, 10, 10, 10),
    (" ",       50, 20, 15, 15),
    (" ",       30, 30, 20, 20),
]

print(f"{' ':<12s} {' data':>8s} {'Wiki':>8s} {'SFT':>8s} {' ':>8s} {' '}")
print("-" * 60)
for name, coarse, wiki, sft, inst in schemes:
    c = decay_total * coarse / 100
    w = decay_total * wiki / 100
    s = decay_total * sft / 100
    i = decay_total * inst / 100
    print(f"{name:<12s} {c:>6.0f}B  {w:>6.0f}B  {s:>6.0f}B  {i:>6.0f}B  ", end="")
    if coarse == 100:
        print("baseline")
    elif coarse >= 50:
        print(" ")
    else:
        print("domain , ")

print()
print(" ：")
print(" 1. data(CC+Code) -> training ")
print(" 2. data 30-50% -> annealing 「 」 ")
print(" 3. data 50% model「 」-> ")

In [ ]:
# === data ===
fig, ax = plt.subplots(figsize=(8, 4))

names = [' ', ' ', ' ', ' ']
coarse = [100, 70, 50, 30]
quality = [0, 30, 50, 70]  #  data 

x = np.arange(len(names))
ax.bar(x, coarse, label='Coarse data (CC+Code)', color='#95a5a6', alpha=0.8)
ax.bar(x, quality, bottom=coarse, label='High-quality data (Wiki+SFT+instructions)', color='#2ecc71', alpha=0.8)
ax.set_ylabel('Share (%)')
ax.set_title('Data mix during decay')
ax.set_xticks(x)
ax.set_xticklabels(names)
ax.legend()
ax.axhline(y=50, color='red', ls='--', alpha=0.5, label='50% reference line')

#  
ax.axvspan(0.5, 2.5, alpha=0.05, color='green')
ax.text(1.5, 55, 'Recommended range', ha='center', fontsize=10, color='green')

plt.tight_layout()
plt.show()


---

## 5. Continual Pretraining (CPT): turn a general model into a domain expert

Annealing and data switching are within pretraining.
A bigger scenario is:

> You have a general LLM (e.g. LLaMA 3). Now you want it to learn domain knowledge (medical / legal / finance).

This is **CPT**.

### 5.1 Core challenge: catastrophic forgetting

```
You study English for 3 years -> score 90 (pretraining)
Then you study only French for 3 months -> French 80, but English drops to 50 (forgetting)
If you keep reviewing English while learning French -> French 80, English 75 (CPT goal)
```

Data view: you must keep some general data as an anchor.


In [ ]:
# === CPT data manual ===
print("=== CPT data ：domaindata vs generaldata ===")
print()

cpt_total = 50  # B tokens

strategies = [
    (" domaindata",    100,   0, " ：model "),
    ("domain ",       70,  30, "general "),
    (" ",       50,  50, " ： "),
    (" ",       30,  70, "domain "),
]

print(f" CPT data: {cpt_total}B tokens")
print()
print(f"{' ':<14s} {'domain':>8s} {'general':>8s} {' '}")
print("-" * 50)
for name, domain, general, note in strategies:
    d = cpt_total * domain / 100
    g = cpt_total * general / 100
    print(f"{name:<14s} {d:>6.0f}B  {g:>6.0f}B  {note}")

print()
print(" ：domaindata : generaldata ≈ 1:1 2:1")
print("generaldata = 「 」, model ")

In [ ]:
# === vs ： ===
fig, ax = plt.subplots(figsize=(8, 4))

domain_pct = np.array([0, 30, 50, 70, 100])
forget_risk = np.array([0, 10, 25, 60, 90])  #  
domain_learn = np.array([0, 40, 75, 90, 95])  # domain 

ax2 = ax.twinx()
ax.plot(domain_pct, forget_risk, 'r-o', lw=2, ms=8, label='Forgetting risk')
ax2.plot(domain_pct, domain_learn, 'b--s', lw=2, ms=8, label='Domain learning')

ax.axvspan(40, 70, alpha=0.1, color='green', label='Recommended range')
ax.axvline(x=60, color='green', ls='--', alpha=0.7, label='Best ~60%')

ax.set_xlabel('Domain data share (%)')
ax.set_ylabel('Forgetting risk (%)', color='red')
ax2.set_ylabel('Domain learning effect (%)', color='blue')
ax.set_title('CPT sweet spot: 50-70% domain data')

lines1, labels1 = ax.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax.legend(lines1 + lines2, labels1 + labels2, fontsize=8, loc='center right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print(" = , = ")
print(" —— .")


### 5.2 LoRA for CPT: less forgetting + less VRAM

Two paths for CPT:

| Method | Trainable params | Forgetting risk | VRAM (7B) |
|:---|:---|:---|:---|
| Full CPT | all 7B | higher (everything moves) | ~120 GB |
| **LoRA CPT** | ~21M (~0.3%) | **lower (base weights frozen)** | **~15 GB** |

LoRA naturally reduces forgetting: W stays fixed, only the side path learns.
It is like adding a small attic on top of a stable foundation.


In [ ]:
# === CPT Compare ===
print("=== CPT Compare(7B model)===")
print()
print(f"{' ':<20s} {' ':>8s} {'trainingparams':>10s} {' ':>10s} {' ':>8s}")
print("-" * 60)

#  CPT
print(f"{' CPT':<20s} {'14 GB':>8s} {'7000M':>10s} {'84 GB':>10s} {'~120 GB':>8s}")

# LoRA CPT
print(f"{'LoRA CPT (r=16)':<20s} {'14 GB':>8s} {'~21M':>10s} {'~1 GB':>10s} {'~15 GB':>8s}")

# QLoRA CPT
print(f"{'QLoRA CPT (4bit)':<20s} {'3.5 GB':>8s} {'~21M':>10s} {'~1 GB':>10s} {'~6 GB':>8s}")

print()
print("QLoRA need 6GB -> RTX 3060 7B model domain CPT！")
print()
print("LoRA CPT ：")
print(" 1. -> -> general ")
print(" 2. AB -> ")
print(" 3. merge -> ( model )")
print()
print("Note： CPT data (>100B),LoRA .")
print(" r( 64 128), CPT.")

---

## 6. Panorama: the full training lifecycle

Connect everything in this part:

```
+-----------------------------------------------------------------+
|                 Full LLM training lifecycle                      |
+-----------------------------------------------------------------+
| 1) Pretraining                                                   |
|    data: Common Crawl + Wiki + Code + Books (10T+ tokens)        |
|    schedule: WSD (stable phase, constant LR)                     |
|                                                                 |
| 2) Mid-training annealing                                        |
|    data: mix in 30-50% high-quality data (Wiki/SFT/instruction)  |
|    schedule: WSD decay phase                                     |
|                                                                 |
| 3) Continual pretraining (optional)                              |
|    data: domain-heavy + some general anchor                      |
|    method: full or LoRA                                          |
|                                                                 |
| 4) SFT                                                           |
| 5) RLHF / DPO                                                    |
+-----------------------------------------------------------------+
```


---

## Checklist

- [ ] Cosine limitation: total steps fixed up front; hard to continue
- [ ] WSD phases: Warmup -> Stable -> Decay
- [ ] WSD advantage: Stable LR constant -> can stop/resume; decay later
- [ ] Why annealing works: direction found + smaller precise steps + better data guidance
- [ ] Mid-training data: mixing 30-50% high-quality during decay can help
- [ ] CPT challenge: catastrophic forgetting
- [ ] CPT mixing rule: domain 50-70% + general 30-50% as anchor
- [ ] LoRA CPT: freeze base weights -> less forgetting + much less VRAM
- [ ] Full lifecycle: pretrain -> anneal -> CPT(optional) -> SFT -> RLHF

One-sentence summary: pretraining is not the end. WSD makes training continuable, annealing makes the last compute count, and CPT turns a base model into a domain expert.
